# Criteo Uplift v2: EDA

Quick sanity checks on the parquet:

1. Row count matches the published 13,979,592.
2. Treatment ratio is ~0.85 (Criteo's RCT assignment).
3. Baseline visit rate per arm.
4. Naive ATE on `visit` is around 0.003.
5. Peak RSS during conversion stays under 8 GB.

In [ ]:
import psutil
import pyarrow.parquet as pq
import pandas as pd

from upliftbench.config import (
    CRITEO_PARQUET,
    CRITEO_ROW_COUNT,
    FEATURES,
    TREATMENT_COL,
    OUTCOME_VISIT,
    OUTCOME_CONVERSION,
)

In [ ]:
pf = pq.ParquetFile(str(CRITEO_PARQUET))
print(f"num_rows: {pf.metadata.num_rows:,}")
print(f"expected: {CRITEO_ROW_COUNT:,}")
assert pf.metadata.num_rows == CRITEO_ROW_COUNT

In [ ]:
cols = [TREATMENT_COL, OUTCOME_VISIT, OUTCOME_CONVERSION]
df = pq.read_table(str(CRITEO_PARQUET), columns=cols).to_pandas()
print(df.describe())
print(f"\ntreatment ratio: {df[TREATMENT_COL].mean():.4f}")
print(f"visit rate, treated:   {df.loc[df[TREATMENT_COL] == 1, OUTCOME_VISIT].mean():.4f}")
print(f"visit rate, control:   {df.loc[df[TREATMENT_COL] == 0, OUTCOME_VISIT].mean():.4f}")
naive_ate_visit = (
    df.loc[df[TREATMENT_COL] == 1, OUTCOME_VISIT].mean()
    - df.loc[df[TREATMENT_COL] == 0, OUTCOME_VISIT].mean()
)
print(f"naive ATE on visit: {naive_ate_visit:+.4f}")
assert abs(naive_ate_visit - 0.0029) < 0.002, naive_ate_visit

In [ ]:
proc = psutil.Process()
print(f"current RSS: {proc.memory_info().rss / 1e9:.2f} GB")

In [ ]:
feat_summary = pd.read_parquet(str(CRITEO_PARQUET), columns=FEATURES).describe().T
feat_summary[['min', 'max', 'mean', 'std']]